In [0]:
from pyspark.sql.functions import col

from functools import reduce

# Here, I first will standarize the column fraud_bool to is_fraud for the three datasets

# For BAF base
baf_gold_base = spark.table("fraud_silver.baf_base")

baf_gold_base_st = (
    baf_gold_base
    .withColumnRenamed("fraud_bool", "is_fraud") 
)
baf_gold_base_st.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("fraud_gold.baf_base_standarized")

# For BAF_variant_3
baf_gold_variant_3 = spark.table("fraud_silver.baf_variant_3")

baf_gold_variant_3_st = (
    baf_gold_variant_3
    .withColumnRenamed("fraud_bool", "is_fraud") 
)

baf_gold_variant_3_st.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("fraud_gold.baf_variant_3_standarized")

# For BAF_variant_5

baf_gold_variant_5 = spark.table("fraud_silver.baf_variant_5")

baf_gold_variant_5_st = (
    baf_gold_variant_5
    .withColumnRenamed("fraud_bool", "is_fraud") 
)

baf_gold_variant_5_st.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("fraud_gold.baf_variant_5_standarized")

# Now, I will unified all three datasets

baf_base = spark.table("fraud_gold.baf_base_standarized")
baf_variant_3 = spark.table("fraud_gold.baf_variant_3_standarized")
baf_variant_5 = spark.table("fraud_gold.baf_variant_5_standarized")


gold_layer_baf = reduce(
    lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True),
    [baf_base, baf_variant_3, baf_variant_5]
)

# And save them on a consolidated table

gold_layer_baf.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("fraud_gold.account_application_fraud")

